# Simulasi Diagnosa Penyakit ML Pipeline
> Arsitektur Machine Learning Pipeline
Pipeline berfungsi membungkus seluruh tahap transformasi data dan model ke dalam satu objek tunggal. Hal ini mencegah terjadinya data leakage (kebocoran data) saat proses validasi.
Komponen Utama:
1. Data Cleaning: Menangani nilai yang hilang (missing values), misalnya mengisi rata-rata pada kolom tekanan darah.
2. Feature Scaling: Menyamakan skala data (misalnya, umur vs. kadar kolesterol) menggunakan StandardScaler agar model tidak bias.
3. Encoding: Mengubah data kategori (seperti jenis kelamin atau riwayat merokok) menjadi angka.
4. Model Estimator: Algoritma klasifikasi seperti Random Forest atau Support Vector Machine (SVM).

### Simulasi Model Pipeline

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
import warnings
warnings.filterwarnings("ignore", category=UserWarning)
import numpy as np

# Mengatur seed agar data yang dihasilkan konsisten
np.random.seed(42)
# Jumlah sampel data
n_samples = 200
# Generate fitur kesehatan
data = {
    'Usia': np.random.randint(15, 80, n_samples),
    'BMI': np.round(np.random.uniform(17.0, 35.0, n_samples), 1),
    'Tekanan_Darah': np.random.randint(90, 180, n_samples),
    'Gula_Darah': np.random.randint(70, 250, n_samples),
    'Aktivitas_Fisik': np.random.choice(['Rendah', 'Sedang', 'Tinggi'], n_samples)
}
df = pd.DataFrame(data)
# Simulasi Logika Diagnosa (Target)
# Seseorang berisiko (1) jika Gula Darah > 140 atau BMI > 30, dengan sedikit variasi acak
conditions = (
    (df['Gula_Darah'] > 140) |
    (df['BMI'] > 30) |
    ((df['Tekanan_Darah'] > 140) & (df['Usia'] > 50))
)
df['Diagnosa'] = conditions.astype(int)
# Menambahkan sedikit 'noise' agar model ML belajar lebih keras
noise = np.random.choice([0, 1], size=n_samples, p=[0.9, 0.1])
df['Diagnosa'] = np.where(noise, 1 - df['Diagnosa'], df['Diagnosa'])
# Simpan ke CSV
df.to_csv('data_kesehatan.csv', index=False)
print("File 'data_kesehatan.csv' berhasil dibuat!")
# 1. Load Data
df = pd.read_csv('data_kesehatan.csv')
X = df.drop('Diagnosa', axis=1)
y = df['Diagnosa']
# 2. Split Data (80% Training, 20% Testing)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
# 3. Definisi Preprocessing
numeric_features = ['Usia', 'BMI', 'Tekanan_Darah', 'Gula_Darah']
categorical_features = ['Aktivitas_Fisik']
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features),
        ('cat', OneHotEncoder(), categorical_features)
    ])
# 4. Membangun Pipeline Lengkap
model_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(n_estimators=100, random_state=42))
])
# 5. Training
model_pipeline.fit(X_train, y_train)
# Dengan menyimpan pipeline (Sekarang setelah model terlatih)
joblib.dump(model_pipeline, 'model_diagnosa_kesehatan.pkl')

File 'data_kesehatan.csv' berhasil dibuat!


['model_diagnosa_kesehatan.pkl']

### Simulasi Prediksi dengan Model Pipeline

In [3]:
import pandas as pd
import joblib
try:
    # Muat ulang model dari file .pkl sebelum prediksi
    model = joblib.load('model_diagnosa_kesehatan.pkl')
except:
    model = None
    print("Peringatan: File 'model_diagnosa_kesehatan.pkl' tidak ditemukan!")

input_df = pd.DataFrame([{
    'Usia': 60,
    'BMI': 40,
    'Tekanan_Darah': 90, # Menambahkan fitur yang hilang
    'Gula_Darah': 150,
    'Aktivitas_Fisik': 'Tinggi'
}])
print(input_df)
try:
    prediction = model.predict(input_df)
    if prediction == 1:
      print("Hasil Diagnosa: Pasien Berisiko Tinggi (Perlu Pemeriksaan Lanjut)")
    else:
      print("Hasil Diagnosa: Kondisi Pasien Normal")
except:
    print("Peringatan: Model tidak dapat melakukan prediksi.")

   Usia  BMI  Tekanan_Darah  Gula_Darah Aktivitas_Fisik
0    60   40             90         150          Tinggi
Hasil Diagnosa: Pasien Berisiko Tinggi (Perlu Pemeriksaan Lanjut)


In [10]:
import ipywidgets as widgets
from IPython.display import display, clear_output
import pandas as pd
import plotly.graph_objects as go # Changed to graph_objects for more control
import plotly.express as px

# Mendapatkan nama kolom fitur dari data pelatihan (X)
feature_columns_for_widgets = X.columns.tolist()

# Membuat widget input untuk setiap fitur
input_widgets = {}
for feature in feature_columns_for_widgets:
    if feature == 'jenis_kelamin':
        input_widgets[feature] = widgets.Dropdown(
            options=[('Pria', 1), ('Wanita', 0)], # 1=Pria, 0=Wanita
            description=feature.replace('_', ' ').title() + ':',
            value=1 # Default: Pria
        )
    elif feature == 'angina_saat_olahraga':
        input_widgets[feature] = widgets.Dropdown(
            options=[('Ya', 1), ('Tidak', 0)], # 1=Ya, 0=Tidak
            description=feature.replace('_', ' ').title() + ':',
            value=0 # Default: Tidak
        )
    elif feature == 'Aktivitas_Fisik': # Handle 'Aktivitas_Fisik' as a categorical feature
        input_widgets[feature] = widgets.Dropdown(
            options=['Rendah', 'Sedang', 'Tinggi'],
            description=feature.replace('_', ' ').title() + ':',
            value='Sedang' # Default value
        )
    else:
        # Menggunakan X untuk mendapatkan median default yang sesuai dengan nama kolom
        default_value = X[feature].median() if feature in X.columns else 0
        input_widgets[feature] = widgets.IntText(
            description=feature.replace('_', ' ').title() + ':',
            value=int(default_value) # Default value as median
        )

# Tombol prediksi
predict_button = widgets.Button(description="Prediksi")

# Area output untuk menampilkan hasil prediksi dan visualisasi
output_prediction = widgets.Output()
output_plot = widgets.Output()

def on_predict_button_clicked(b):
    with output_prediction:
        clear_output()
        new_patient_input = {feature: widget.value for feature, widget in input_widgets.items()}

        print("Data Pasien Baru dari Input Form:")
        for k, v in new_patient_input.items():
            print(f"  {k.replace('_', ' ').title()}: {v}")

        # Convert new_patient_input to DataFrame for prediction
        new_patient_df = pd.DataFrame([new_patient_input], columns=feature_columns_for_widgets)

        # Perform prediction
        prediction = model.predict(new_patient_df)
        prediction_proba = model.predict_proba(new_patient_df)

        result_text = ""
        if prediction[0] == 1:
            result_text = f"Pasien diprediksi memiliki **Penyakit Jantung** (Probabilitas: {prediction_proba[0][1]*100:.2f}%)"
        else:
            result_text = f"Pasien diprediksi **Tidak Memiliki Penyakit Jantung** (Probabilitas: {prediction_proba[0][0]*100:.2f}%)"

        print(f"\nHasil Prediksi: {result_text}")

    with output_plot:
        clear_output()
        # Create a DataFrame for probabilities
        proba_df = pd.DataFrame({
            'Class': ['Tidak Memiliki Penyakit Jantung', 'Penyakit Jantung'],
            'Probability': prediction_proba[0]
        })
        display(proba_df)   
        # Create an interactive bar chart for prediction probabilities
        fig_proba = go.Figure(data=[go.Bar(
            x=proba_df['Class'],
            y=proba_df['Probability'],
            marker_color=['green', 'red'] if prediction[0] == 0 else ['red', 'green']
        )])

        fig_proba.update_layout(
            title='<b>Probabilitas Prediksi Penyakit Jantung</b>',
            xaxis_title='Kondisi',
            yaxis_title='Probabilitas',
            yaxis_range=[0, 1.05], # Set y-axis range from 0 to 1
            template='plotly_white'
        )
        fig_proba.show()
        
        

predict_button.on_click(on_predict_button_clicked)

# Tampilkan semua widget
print("Masukkan Data Pasien Baru:")
for feature in feature_columns_for_widgets:
    display(input_widgets[feature])
display(predict_button, output_prediction, output_plot)

Masukkan Data Pasien Baru:


IntText(value=47, description='Usia:')

IntText(value=26, description='Bmi:')

IntText(value=132, description='Tekanan Darah:')

IntText(value=148, description='Gula Darah:')

Dropdown(description='Aktivitas Fisik:', index=1, options=('Rendah', 'Sedang', 'Tinggi'), value='Sedang')

Button(description='Prediksi', style=ButtonStyle())

Output()

Output()